# A2 AMP Results — Jurisdiction-Held-Out Test Sets

This notebook displays canonical outputs from `src/experiments/11_evaluate_amp.py`. It does not recompute metrics or bootstrap intervals. A2 uses SHERLOC Legacy Keywords as a **silver reference**. Where the pooled A2 reference support for `PURPOSE_REMOVAL_OF_ORGANS` is zero, its per-label F1 is **N/A** and macro-F1 is calculated over the 16 supported labels by the evaluator; all 17 outputs still enter micro/set metrics.

No scientific conclusions or protocol changes are generated here.

## 1. Setup and canonical-artifact contract

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

EXPECTED_METHODS = ("M1", "M2", "M3", "M4")


def locate_repo_root() -> Path:
    """Locate the repository without relying on the notebook launch directory."""
    configured = os.environ.get("SHERLOC_REPO_ROOT")
    starts = [Path(configured).expanduser()] if configured else []
    starts.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in starts:
        if (candidate / "src/experiments/11_evaluate_amp.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate SHERLOC_Case_Analysis. Start Jupyter in the repository "
        "or set SHERLOC_REPO_ROOT."
    )


REPO_ROOT = locate_repo_root()
METRICS_ROOT = REPO_ROOT / "outputs/metrics"


def load_json(path: Path) -> dict:
    if not path.is_file():
        display(Markdown(f"> **Pending:** `{path.relative_to(REPO_ROOT)}` does not exist."))
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def load_csv(path: Path, required_columns=()) -> pd.DataFrame:
    """Load an evaluator table, reporting absence without synthesizing results."""
    if not path.is_file():
        display(Markdown(f"> **Pending:** `{path.relative_to(REPO_ROOT)}` does not exist."))
        return pd.DataFrame(columns=list(required_columns))
    frame = pd.read_csv(path)
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing canonical columns: {sorted(missing)}")
    return frame


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def show_or_pending(frame: pd.DataFrame, message="No canonical rows are available yet."):
    if frame.empty:
        display(Markdown(f"> **Pending:** {message}"))
    else:
        display(frame)


def methods_complete(manifest: dict, evaluation: str) -> bool:
    methods = manifest.get("evaluations", {}).get(evaluation, {}).get("methods", [])
    return set(methods) == set(EXPECTED_METHODS)


manifest_path = METRICS_ROOT / "amp_evaluation_manifest.json"
evaluation_manifest = load_json(manifest_path)


## 2. Artifact availability and completion gate

In [ ]:
canonical_inputs = [
    METRICS_ROOT / "amp_evaluation_manifest.json",
    METRICS_ROOT / "a1/amp_primary_results.csv",
    METRICS_ROOT / "a1/amp_per_label.csv",
    METRICS_ROOT / "a1/amp_bootstrap_cis.csv",
    METRICS_ROOT / "a1/amp_case_level_errors.csv",
    METRICS_ROOT / "a2/amp_primary_results.csv",
    METRICS_ROOT / "a2/amp_per_fold.csv",
    METRICS_ROOT / "a2/amp_per_label.csv",
    METRICS_ROOT / "a2/amp_per_jurisdiction.csv",
    METRICS_ROOT / "a2/amp_bootstrap_cis.csv",
    METRICS_ROOT / "a2/amp_case_level_errors.csv",
    METRICS_ROOT / "amp_a1_to_a2_deltas.csv",
]
availability = pd.DataFrame(
    {
        "artifact": [str(path.relative_to(REPO_ROOT)) for path in canonical_inputs],
        "available": [path.is_file() for path in canonical_inputs],
    }
)
display(availability)

gate = evaluation_manifest.get("final_completion_gate", "PENDING")
complete = (
    gate == "PASSED_M1_M2_M3_M4_A1_A2"
    and methods_complete(evaluation_manifest, "A1")
    and methods_complete(evaluation_manifest, "A2")
)
if complete:
    display(Markdown("**Canonical completion gate: PASSED for M1-M4 in A1 and A2.**"))
else:
    display(Markdown(
        "> **Incomplete benchmark:** canonical M1-M4 A1/A2 outputs are not complete. "
        "Any available rows are technical previews, not the final comparison."
    ))


## 3. A2 fold composition and held-out jurisdictions

In [ ]:
a2_split_path = REPO_ROOT / "data/splits/a2_jurisdiction_folds_final_v1.csv"
a2_split = load_csv(a2_split_path, ("search_rank", "fold_id", "role", "jurisdiction", "heldout_jurisdiction"))
if not a2_split.empty:
    fold_composition = (
        a2_split.groupby(["fold_id", "role"], dropna=False)
        .agg(cases=("search_rank", "size"), jurisdictions=("jurisdiction", "nunique"))
        .reset_index()
    )
    display(fold_composition)
    heldout = (
        a2_split.loc[a2_split["role"].eq("TEST"), ["fold_id", "jurisdiction"]]
        .drop_duplicates()
        .sort_values(["fold_id", "jurisdiction"])
    )
    display(heldout)
    display(pd.DataFrame([{"split_sha256": sha256_file(a2_split_path)}]))
else:
    show_or_pending(a2_split)


## 4. Pooled OOD label support and Organ Removal status

In [ ]:
a2_per_label = load_csv(
    METRICS_ROOT / "a2/amp_per_label.csv",
    ("method", "label_id", "family", "support", "precision", "recall", "f1", "status", "included_in_macro_f1"),
)
if not a2_per_label.empty:
    support_view = a2_per_label[["method", "label_id", "family", "support", "status", "included_in_macro_f1"]]
    display(support_view)
    organ_rows = a2_per_label.loc[a2_per_label["label_id"].eq("PURPOSE_REMOVAL_OF_ORGANS")]
    display(Markdown("**Organ Removal evaluator rows:**"))
    display(organ_rows)
    display(pd.DataFrame([{
        "manifest_rule": evaluation_manifest.get("evaluations", {}).get("A2", {}).get("organ_removal_rule", "PENDING"),
        "macro_label_count": evaluation_manifest.get("evaluations", {}).get("A2", {}).get("macro_label_count", "PENDING"),
    }]))
else:
    show_or_pending(a2_per_label)


## 5. Canonical per-fold results

In [ ]:
a2_per_fold = load_csv(
    METRICS_ROOT / "a2/amp_per_fold.csv",
    ("method", "fold", "macro_f1", "micro_f1", "exact_set_accuracy", "example_jaccard", "test_n"),
)
show_or_pending(a2_per_fold)


## 6. Canonical pooled OOD results

In [ ]:
a2_primary = load_csv(
    METRICS_ROOT / "a2/amp_primary_results.csv",
    ("method", "fold_1_macro_f1", "fold_2_macro_f1", "fold_3_macro_f1", "pooled_ood_macro_f1", "pooled_micro_f1", "pooled_exact_set_accuracy", "pooled_example_jaccard", "test_n"),
)
if not a2_primary.empty:
    order = {method: index for index, method in enumerate(EXPECTED_METHODS)}
    a2_primary = a2_primary.assign(_order=a2_primary["method"].map(order)).sort_values("_order").drop(columns="_order")
show_or_pending(a2_primary)


## 7. Canonical per-jurisdiction results

In [ ]:
a2_per_jurisdiction = load_csv(
    METRICS_ROOT / "a2/amp_per_jurisdiction.csv",
    ("method", "jurisdiction", "fold", "macro_f1", "micro_f1", "exact_set_accuracy", "example_jaccard", "test_n"),
)
show_or_pending(a2_per_jurisdiction)


## 8. Canonical A1 → A2 aggregate deltas

In [ ]:
a1_a2_deltas = load_csv(
    METRICS_ROOT / "amp_a1_to_a2_deltas.csv",
    ("method", "delta_macro_f1_a2_minus_a1", "delta_micro_f1_a2_minus_a1", "delta_exact_set_a2_minus_a1", "delta_example_jaccard_a2_minus_a1", "significance_claim"),
)
show_or_pending(a1_a2_deltas)


## 9. Per-label IID/OOD comparison

In [ ]:
a1_per_label = load_csv(
    METRICS_ROOT / "a1/amp_per_label.csv",
    ("method", "label_id", "f1", "support", "status"),
)
if not a1_per_label.empty and not a2_per_label.empty:
    # This is a side-by-side view of already-computed evaluator values. It does
    # not reconstruct F1 or assert statistical significance.
    per_label_comparison = a1_per_label[["method", "label_id", "f1", "support", "status"]].merge(
        a2_per_label[["method", "label_id", "f1", "support", "status"]],
        on=["method", "label_id"], how="outer", suffixes=("_a1", "_a2"), validate="one_to_one",
    )
    display(per_label_comparison)
else:
    display(Markdown("> **Pending:** both canonical A1 and A2 per-label tables are required."))


## 10. Canonical pooled bootstrap confidence intervals

In [ ]:
a2_bootstrap = load_csv(
    METRICS_ROOT / "a2/amp_bootstrap_cis.csv",
    ("method", "metric", "estimate", "ci_lower", "ci_upper", "n_resamples", "seed"),
)
show_or_pending(a2_bootstrap)


## 11. Descriptive visual summaries

In [ ]:
if not a2_primary.empty:
    pooled_columns = ["pooled_ood_macro_f1", "pooled_micro_f1", "pooled_exact_set_accuracy", "pooled_example_jaccard"]
    a2_primary.set_index("method")[pooled_columns].plot.bar(
        subplots=True, layout=(2, 2), figsize=(12, 8), legend=False, ylim=(0, 1),
        title=["Pooled macro-F1", "Pooled micro-F1", "Pooled exact-set", "Pooled Jaccard"],
    )
    plt.suptitle("A2 canonical evaluator metrics (descriptive only)")
    plt.tight_layout()
else:
    display(Markdown("> **Pending:** no canonical A2 aggregate table to plot."))


## 12. Technical observations for researcher review

Record short technical observations only after the completion gate passes. A negative A1→A2 delta is not automatically statistically significant; the canonical delta table explicitly records `NOT_TESTED_DO_NOT_INFER`. Do not change later folds, prompts, demonstrations, models, or thresholds based on these test outcomes.